# Protocol resting

## Import raw data to structured JSON

Import raw resting data from `datasets/raw/resting/` to structured JSON in `datasets/resting/`.
Drop your `.txt` / `.csv` Polar export files into `cardiolab/datasets/raw/resting/` before running this cell.

In [ ]:
from cardiolab.scripts.import_rr import import_all

if __name__ == "__main__":
    import_all(protocol="resting")

## Pipeline Score, RMSSD, ... from json file

Print for all file json export with the date, and the differents stats from time domain (RMSSD, SDNN, PNN50 and mean HR)

In [ ]:
# from __future__ import annotations

import glob
import json

from cardiolab.analytics.baseline import Baseline
from cardiolab.protocols.resting import HRVFeatures, resting_hrv
from cardiolab.signals.rr import RRSeries

# from cardiolab.analytics.scoring import readiness_score_oura


path = "cardiolab/datasets/resting/*.json"

files = sorted(glob.glob(path))
features_list = []

dates = []
rmssd_values = []
scores = []

baseline = Baseline()

for file in files:
    with open(file) as f:
        data = json.load(f)

    rr = RRSeries(data["rr_intervals"])
    result = resting_hrv(rr)

    features = HRVFeatures(
        date=data["date"],
        rmssd=result.rmssd,
        ln_rmssd=result.ln_rmssd,
        sdnn=result.sdnn,
        pnn50=result.pnn50,
        mean_hr=result.mean_hr,
        vlf=result.vlf,
        lf=result.lf,
        hf=result.hf,
        lf_hf=result.lf_hf,
        hf_pct=result.hf_pct,
        lf_nu=result.lf_nu,
        hf_nu=result.hf_nu,
        hf_hr=result.hf_hr,
    )

    print(
        {
            "date": features.date,
            "rmssd_value": features.rmssd,
            "sdnn_value": features.sdnn,
            "pnn50_value": features.pnn50,
            "mean_hr_value": features.mean_hr,
            "vlf": features.vlf,
            "lf": features.lf,
            "hf": features.hf,
            "lf_hf": features.lf_hf,
            "hf_pct": features.hf_pct,
            "lf_nu": features.lf_nu,
            "hf_nu": features.hf_nu,
            "hf_hr": features.hf_hr,
        }
    )

    # features_list.append(features)

In [ ]:
# from __future__ import annotations

import glob
import json

from cardiolab.analytics.baseline import Baseline
from cardiolab.analytics.scoring import readiness_score_multi
from cardiolab.protocols.resting import HRVFeatures, resting_hrv
from cardiolab.signals.rr import RRSeries


def load_features_from_json(
    path="cardiolab/datasets/resting/*.json",
) -> list[HRVFeatures]:
    """Load resting protocol JSON files and convert them into HRVFeatures."""
    files = sorted(glob.glob(path))
    features_list = []

    for file in files:
        with open(file) as f:
            data = json.load(f)

        rr = RRSeries(data["rr_intervals"])
        result = resting_hrv(rr)

        features = HRVFeatures(
            date=data["date"],
            rmssd=result.rmssd,
            ln_rmssd=result.ln_rmssd,
            sdnn=result.sdnn,
            pnn50=result.pnn50,
            mean_hr=result.mean_hr,
            vlf=result.vlf,
            lf=result.lf,
            hf=result.hf,
            lf_hf=result.lf_hf,
            hf_pct=result.hf_pct,
            lf_nu=result.lf_nu,
            hf_nu=result.hf_nu,
            hf_hr=result.hf_hr,
        )

        features_list.append(features)

    return features_list


def compute_scores(features_list: list[HRVFeatures], baseline: Baseline):
    """Compute and assign score to each feature."""
    for f in features_list:
        f.score = readiness_score_multi(f, baseline)


def display_results(features_list: list[HRVFeatures], baseline: Baseline):
    """Display full results."""
    print("\n===== DAILY METRICS =====\n")

    for f in features_list:
        print(f"Date    : {f.date}")
        print(f"RMSSD   : {f.rmssd:.2f} ms")
        print(f"HR      : {f.mean_hr:.2f} bpm")
        print(f"HF      : {f.hf:.5f}")
        print(f"HF/FC   : {f.hf_hr:.2f}")
        print(f"Score   : {f.score:.2f}")
        print("-" * 30)

    print("\n===== BASELINE =====\n")
    print("Mean RMSSD:", baseline.mean_rmssd())
    print("Median RMSSD:", baseline.median_rmssd())
    print("Mean HR:", baseline.mean_hr())


def run():
    """Full resting pipeline."""
    features_list = load_features_from_json()

    if not features_list:
        print("No data found.")
        return

    baseline = Baseline.from_features(features_list)
    compute_scores(features_list, baseline)
    display_results(features_list, baseline)


if __name__ == "__main__":
    run()

| Score  | Interprétation     |
| ------ | ------------------ |
| 80–100 | très bien récupéré |
| 60–80  | ok                 |
| 40–60  | fatigue modérée    |
| 20–40  | fatigué            |
| <20    | surcharge          |

In [ ]:
from cardiolab.visualization.plot import plot_resting_evolution

plot_resting_evolution(path="cardiolab/datasets/resting/*.json")

# Protocol orthostatic

## Import raw data to structured JSON

Import raw orthostatic data from `datasets/raw/orthostatic/` to structured JSON in `datasets/orthostatic/`.
The recording must contain a supine phase immediately followed by a standing phase (≥ 10 min total recommended).
Drop your `.txt` / `.csv` Polar export files into `cardiolab/datasets/raw/orthostatic/` before running this cell.

In [ ]:
from cardiolab.scripts.import_rr import import_all

if __name__ == "__main__":
    import_all(protocol="orthostatic")

## Pipeline — affichage rapide par session

Pour chaque fichier JSON, exécute le protocole orthostatic et affiche :
- Phase allongé : tous les indicateurs HRV (RMSSD, SDNN, pNN50, HR, VLF/LF/HF…)
- Transition : delta HR, HR max, durée, timestamps
- Phase debout : tous les indicateurs HRV
- Résumé : réponse HR, variation LF/HF, variation HF%, interprétation clinique

In [ ]:
import glob
import json

from cardiolab.protocols.orthostatic import orthostatic_hrv
from cardiolab.signals.rr import RRSeries

path = "cardiolab/datasets/orthostatic/*.json"
files = sorted(glob.glob(path))

for file in files:
    with open(file) as f:
        data = json.load(f)

    rr = RRSeries(data["rr_intervals"])

    try:
        result = orthostatic_hrv(rr)
    except ValueError as e:
        print(f"[SKIP] {data['date']}: {e}")
        continue

    sup = result.phases.supine.features
    std = result.phases.standing.features
    tr = result.phases.transition

    print(f"\n{'=' * 55}")
    print(f"Date: {data['date']}")

    print("\n--- PHASE ALLONGÉ ---")
    print(f"  Durée    : {result.phases.supine.duration_sec:.0f}s")
    print(f"  RMSSD    : {sup.rmssd:.2f} ms")
    print(f"  SDNN     : {sup.sdnn:.2f} ms")
    print(f"  pNN50    : {sup.pnn50:.1f} %")
    print(f"  HR moyen : {sup.mean_hr:.1f} bpm")
    print(f"  HF       : {sup.hf:.5f}")
    print(f"  HF/FC    : {sup.hf_hr:.2f}")
    print(f"  LF/HF    : {sup.lf_hf:.2f}")
    print(f"  HF_nu    : {sup.hf_nu:.2f}")
    print(f"  LF_nu    : {sup.lf_nu:.2f}")

    print("\n--- TRANSITION ---")
    print(f"  Durée    : {tr.duration_sec:.1f}s")
    print(f"  Delta HR : +{tr.delta_hr:.1f} bpm")
    print(f"  HR max   : {tr.peak_hr:.1f} bpm")
    print(f"  Début    : {tr.start_sec:.0f}s  /  Fin: {tr.end_sec:.0f}s")

    print("\n--- PHASE DEBOUT ---")
    print(f"  Durée    : {result.phases.standing.duration_sec:.0f}s")
    print(f"  RMSSD    : {std.rmssd:.2f} ms")
    print(f"  SDNN     : {std.sdnn:.2f} ms")
    print(f"  pNN50    : {std.pnn50:.1f} %")
    print(f"  HR moyen : {std.mean_hr:.1f} bpm")
    print(f"  HF       : {std.hf:.5f}")
    print(f"  HF/FC    : {std.hf_hr:.2f}")
    print(f"  LF/HF    : {std.lf_hf:.2f}")
    print(f"  HF_nu    : {std.hf_nu:.2f}")
    print(f"  LF_nu    : {std.lf_nu:.2f}")

    print("\n--- RÉSUMÉ ORTHOSTATIQUE ---")
    print(f"  Réponse HR  : +{result.hr_response:.1f} bpm")
    print(f"  Δ LF/HF     : x{result.lf_hf_ratio_change:.2f}")
    print(f"  Δ HF        : {result.hf_response_pct:.1f}%")
    print(f"  Δ HF/FC%    : {result.hf_hr_pct_change:.1f}%")
    print(f"  Résultat    : {result.interpretation}")

## Pipeline complet (fonctions)

In [ ]:
import glob
import json
from dataclasses import dataclass

from cardiolab.protocols.orthostatic import OrthostaticResult, orthostatic_hrv
from cardiolab.signals.rr import RRSeries


@dataclass
class OrthostaticSession:
    """Container for a single orthostatic session."""

    date: str
    result: OrthostaticResult


def load_orthostatic_sessions(
    path: str = "cardiolab/datasets/orthostatic/*.json",
) -> list[OrthostaticSession]:
    """Load orthostatic JSON files and run the orthostatic HRV protocol."""
    files = sorted(glob.glob(path))
    sessions = []

    for file in files:
        with open(file) as f:
            data = json.load(f)

        rr = RRSeries(data["rr_intervals"])

        try:
            result = orthostatic_hrv(rr)
            sessions.append(OrthostaticSession(date=data["date"], result=result))
        except ValueError as e:
            print(f"[SKIP] {data['date']}: {e}")

    return sessions


def _display_hrv_features(label: str, features, duration_sec: float) -> None:
    """Display all HRVFeatures fields for a given phase."""
    print(f"\n  {label} (durée: {duration_sec:.0f}s)")
    print(f"    RMSSD    : {features.rmssd:.2f} ms")
    print(f"    ln_RMSSD : {features.ln_rmssd:.3f}")
    print(f"    SDNN     : {features.sdnn:.2f} ms")
    print(f"    pNN50    : {features.pnn50:.1f} %")
    print(f"    HR moyen : {features.mean_hr:.1f} bpm")
    print(f"    VLF      : {features.vlf:.5f}")
    print(f"    LF       : {features.lf:.5f}")
    print(f"    HF       : {features.hf:.5f}")
    print(f"    HF/FC    : {features.hf_hr:.2f}")
    print(f"    LF/HF    : {features.lf_hf:.2f}")
    print(f"    HF%      : {features.hf_pct:.2f}")
    print(f"    LF_nu    : {features.lf_nu:.2f}")
    print(f"    HF_nu    : {features.hf_nu:.2f}")


def display_orthostatic_results(sessions: list[OrthostaticSession]) -> None:
    """Display full HRV features for all sessions."""
    for session in sessions:
        r = session.result
        phases = r.phases

        print(f"\n{'=' * 55}")
        print(f"  Date: {session.date}")
        print(f"{'=' * 55}")

        _display_hrv_features(
            "ALLONGÉ", phases.supine.features, phases.supine.duration_sec
        )

        print(f"\n  TRANSITION (durée: {phases.transition.duration_sec:.1f}s)")
        print(f"    Delta HR : +{phases.transition.delta_hr:.1f} bpm")
        print(f"    HR max   : {phases.transition.peak_hr:.1f} bpm")
        print(f"    Début    : {phases.transition.start_sec:.0f}s")
        print(f"    Fin      : {phases.transition.end_sec:.0f}s")

        _display_hrv_features(
            "DEBOUT", phases.standing.features, phases.standing.duration_sec
        )

        print("\n  RÉSUMÉ ORTHOSTATIQUE")
        print(f"    Réponse HR  : +{r.hr_response:.1f} bpm")
        print(f"    Δ LF/HF     : x{r.lf_hf_ratio_change:.2f}")
        print(f"    Δ HF        : {r.hf_response_pct:.1f}%")
        print(f"    Δ HF/FC%    : {r.hf_hr_pct_change:.1f}%")
        print(f"    Résultat    : {r.interpretation}")


def run_orthostatic() -> None:
    """Full orthostatic pipeline."""
    sessions = load_orthostatic_sessions()

    if not sessions:
        print("No orthostatic data found in cardiolab/datasets/orthostatic/")
        return

    display_orthostatic_results(sessions)


if __name__ == "__main__":
    run_orthostatic()

| Résultat | Signification |
| --- | --- |
| `normal` | Réponse orthostatique normale — hausse HR 5–30 bpm |
| `elevated_response` | Réponse excessive > 30 bpm — possible POTS |
| `impaired_response` | Réponse insuffisante < 5 bpm — possible dysautonomie |
| `excessive_vagal_withdrawal` | Retrait vagal excessif — chute HF > 60 % |